# Проверка окружения: DuckDB + Spatial

Выполните **Run → Run All Cells**. Все ячейки должны завершиться без ошибок.

In [ ]:
import platform
import sys
from pathlib import Path

print("Python:", sys.version)
print("Исполняемый файл:", sys.executable)
print("Windows:", platform.platform())
print("Рабочая папка:", Path.cwd())

In [ ]:
import duckdb
import pandas as pd

print("DuckDB:", duckdb.__version__)
print("pandas:", pd.__version__)

## Установка и загрузка Spatial

При первом запуске `INSTALL` скачивает расширение из официального репозитория DuckDB.

In [ ]:
con = duckdb.connect(":memory:")
con.execute("INSTALL spatial")
con.execute("LOAD spatial")

con.sql("""
    SELECT extension_name, installed, loaded
    FROM duckdb_extensions()
    WHERE extension_name = 'spatial'
""").df()

## Первый пространственный запрос

In [ ]:
cities = con.sql("""
    WITH cities(city, longitude, latitude) AS (
        VALUES
            ('Moscow', 37.6176, 55.7558),
            ('Kazan', 49.1064, 55.7961)
    )
    SELECT
        city,
        longitude,
        latitude,
        ST_AsText(ST_Point(longitude, latitude)) AS geometry_wkt
    FROM cities
""").df()

cities

## Проверка файловой базы

In [ ]:
database_path = Path("data/processed/course.duckdb")
database_path.parent.mkdir(parents=True, exist_ok=True)

file_con = duckdb.connect(str(database_path))
file_con.execute("CREATE OR REPLACE TABLE cities AS SELECT * FROM cities")
rows = file_con.sql("SELECT count(*) AS row_count FROM cities").df()
file_con.close()
con.close()

print("База:", database_path.resolve())
rows

## Готово

Если выше показано `row_count = 2`, окружение готово к прохождению курса.